<a href="https://colab.research.google.com/github/2000030914/2000030914/blob/main/TERM_PAPER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install opencv-python pandas seaborn scikit-learn tensorflow

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras import layers, models

from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
csv_path = "/content/drive/MyDrive/CirrMRI_dataset/T2_age_gender_evaluation.csv"

df = pd.read_csv(csv_path)

label_dict = dict(zip(df["Patient ID"].astype(str), df["Radiological Evaluation"]))

print("Total patients:", len(label_dict))

Total patients: 318


In [ ]:
BASE_PATH = "/content/drive/MyDrive/CirrMRI_dataset/Cirrhosis_T2_2D"

train_path = os.path.join(BASE_PATH, "train")
val_path   = os.path.join(BASE_PATH, "valid")
test_path  = os.path.join(BASE_PATH, "test")

In [ ]:
def count_images(base_path):
    total = 0

    for patient_id in os.listdir(base_path):
        img_folder = os.path.join(base_path, patient_id, "images")

        if not os.path.exists(img_folder):
            continue

        num_images = len([
            f for f in os.listdir(img_folder)
            if f.endswith(('.png', '.jpg', '.jpeg'))
        ])

        total += num_images

    print(f"{base_path} → {total} images")
    return total

count_images(train_path)
count_images(val_path)
count_images(test_path)

/content/drive/MyDrive/CirrMRI_dataset/Cirrhosis_T2_2D/train → 5429 images
/content/drive/MyDrive/CirrMRI_dataset/Cirrhosis_T2_2D/valid → 674 images
/content/drive/MyDrive/CirrMRI_dataset/Cirrhosis_T2_2D/test → 664 images


664

data generator

In [ ]:
class DataGenerator(tf.keras.utils.Sequence):

    def __init__(self, base_path, label_dict, batch_size=32, img_size=224, shuffle=True):
        self.base_path = base_path
        self.label_dict = label_dict
        self.batch_size = batch_size
        self.img_size = img_size
        self.shuffle = shuffle

        self.image_paths = []
        self.labels = []

        for patient_id in os.listdir(base_path):
            img_folder = os.path.join(base_path, patient_id, "images")

            if not os.path.exists(img_folder):
                continue

            if patient_id not in label_dict:
                continue

            label = label_dict[patient_id] - 1

            for img_name in os.listdir(img_folder):
                if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    continue

                img_path = os.path.join(img_folder, img_name)
                self.image_paths.append(img_path)
                self.labels.append(label)

        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.image_paths) / self.batch_size))

    def __getitem__(self, index):
        batch_paths = self.image_paths[index*self.batch_size:(index+1)*self.batch_size]
        batch_labels = self.labels[index*self.batch_size:(index+1)*self.batch_size]

        X, y = [], []

        for img_path, label in zip(batch_paths, batch_labels):
            img = cv2.imread(img_path)

            if img is None:
                continue

            img = cv2.resize(img, (self.img_size, self.img_size))
            img = img.astype(np.float32) / 255.0

            X.append(img)
            y.append(label)

        if len(X) == 0:
            return self.__getitem__((index + 1) % self.__len__())

        return np.array(X), np.array(y)

    def on_epoch_end(self):
        if self.shuffle:
            combined = list(zip(self.image_paths, self.labels))
            np.random.shuffle(combined)
            self.image_paths, self.labels = map(list, zip(*combined))

In [ ]:
train_gen = DataGenerator(train_path, label_dict, batch_size=32, shuffle=True)
val_gen   = DataGenerator(val_path, label_dict, batch_size=32, shuffle=False)
test_gen  = DataGenerator(test_path, label_dict, batch_size=32, shuffle=False)

print("Train batches:", len(train_gen))
print("Validation batches:", len(val_gen))
print("Test batches:", len(test_gen))

Train batches: 168
Validation batches: 22
Test batches: 21


In [ ]:
num_classes = len(set(label_dict.values()))
print("Number of classes:", num_classes)

Number of classes: 3


In [ ]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    # 🔥 Replace Flatten
    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(num_classes, activation='softmax')
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_9 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 222, 222, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 109, 109, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 52, 52, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 111,043 (433.76 KB)

 Trainable params: 110,595 (432.01 KB)

 Non-trainable params: 448 (1.75 KB)

In [ ]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10   # keep small for fast output
)